In [2]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FastEmotionDataset(Dataset):
    def __init__(self, cropped_base, split='train', transform=None):
        self.transform = transform
        self.samples = []
        self.label_map = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4}
        
        split_dir = os.path.join(cropped_base, split)
        
        # 가공된 폴더 구조를 그대로 읽어옴
        for emotion_folder in os.listdir(split_dir):
            if emotion_folder not in self.label_map: continue
            
            folder_path = os.path.join(split_dir, emotion_folder)
            for fname in os.listdir(folder_path):
                self.samples.append({
                    'img_path': os.path.join(folder_path, fname),
                    'label': self.label_map[emotion_folder]
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # 가볍게 잘려있는 이미지 바로 로드
        image = cv2.imread(item['img_path'])
        
        # 만약 비정상 파일이 있다면 예외 처리 (무한루프 방지를 위해 인덱스 0 반환)
        if image is None:
            return self.__getitem__(0)
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
            
        return image, torch.tensor(item['label'], dtype=torch.long)

# 이미 224x224로 잘려있지만, 혹시 모를 크기 불일치를 대비해 Resize 포함
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 새로 만든 데이터셋 경로 지정
CROPPED_BASE = "/workspace/user1/dataset_v3_cropped"
train_dataset = FastEmotionDataset(CROPPED_BASE, split='train', transform=train_transform)
val_dataset = FastEmotionDataset(CROPPED_BASE, split='val', transform=val_transform)

# A40 절반 자원 사용 (Batch 128, Workers 4)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f"⚡ 데이터 로더 세팅 완료! Train: {len(train_dataset)} / Val: {len(val_dataset)}")

⚡ 데이터 로더 세팅 완료! Train: 5882 / Val: 1163


In [3]:
class EmotionClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super(EmotionClassifier, self).__init__()
        # Pretrained 가중치 로드
        self.backbone = models.efficientnet_b2(weights='IMAGENET1K_V1')
        
        # 분류 레이어 5클래스로 변경
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmotionClassifier(num_classes=5).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# VRAM 절약 및 속도 향상을 위한 PyTorch 최신 AMP Scaler
scaler = torch.amp.GradScaler('cuda')

print("✅ 모델 및 옵티마이저 세팅 완료!")

✅ 모델 및 옵티마이저 세팅 완료!


In [4]:
from tqdm import tqdm

num_epochs = 30
best_val_loss = float('inf')
patience = 5
trigger_times = 0

save_dir = '/workspace/user1/model'
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'best_emotion_model.pth')

print("🚀 학습 시작...")

for epoch in range(num_epochs):
    # --- Train Phase ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    train_pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Train")
    for images, labels in train_pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train * 100

    # --- Validation Phase ---
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Val  ")
        for images, labels in val_pbar:
            images, labels = images.to(device), labels.to(device)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_dataset)
    val_acc = correct_val / total_val * 100

    print(f"📊 Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    # --- Early Stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  💾 성능 개선! 모델 저장됨 (Val Loss: {val_loss:.4f})")
        trigger_times = 0
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"\n🛑 {patience} Epoch 동안 개선이 없어 조기 종료합니다.")
            break

print(f"\n✅ 최종 모델 저장 위치: {save_path}")

🚀 학습 시작...


Epoch [1/30] Val  : 100%|██████████| 10/10 [00:10<00:00,  1.01s/it]


📊 Train Loss: 1.2821, Acc: 53.04% | Val Loss: 1.0410, Acc: 66.90%
  💾 성능 개선! 모델 저장됨 (Val Loss: 1.0410)


Epoch [2/30] Val  : 100%|██████████| 10/10 [00:03<00:00,  2.61it/s]


📊 Train Loss: 0.9052, Acc: 74.24% | Val Loss: 0.9700, Acc: 69.65%
  💾 성능 개선! 모델 저장됨 (Val Loss: 0.9700)


Epoch [3/30] Val  : 100%|██████████| 10/10 [00:05<00:00,  1.92it/s]


... (5 outputs left out)

Epoch [6/30] Val  : 100%|██████████| 10/10 [00:05<00:00,  1.95it/s]


📊 Train Loss: 0.5601, Acc: 92.37% | Val Loss: 1.0421, Acc: 72.57%


Epoch [7/30] Val  : 100%|██████████| 10/10 [00:04<00:00,  2.08it/s]

📊 Train Loss: 0.5218, Acc: 94.36% | Val Loss: 1.0476, Acc: 71.28%

🛑 5 Epoch 동안 개선이 없어 조기 종료합니다.

✅ 최종 모델 저장 위치: /workspace/user1/model/best_emotion_model.pth


In [ ]:
# 이미지 crop
'''import os
import json
import cv2
import numpy as np
from tqdm import tqdm

INPUT_BASE = "/workspace/user1/dataset_v2"
OUTPUT_BASE = "/workspace/user1/dataset_v3_cropped"
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("✂️ 데이터 영구 크롭 재작업 시작 (정밀 추적 모드)...")

success_count = 0
error_count = 0

for split in ['train', 'val']:
    lbl_base = os.path.join(INPUT_BASE, 'label_mod', split)
    img_base = os.path.join(INPUT_BASE, 'img', split)
    npz_base = os.path.join(INPUT_BASE, 'segmentation', split)
    
    if not os.path.exists(lbl_base): continue
    
    for json_file in os.listdir(lbl_base):
        if not json_file.endswith('.json'): continue
        
        with open(os.path.join(lbl_base, json_file), 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        images_list = data['images']
        
        current_npz_path = ""
        npz_data = None
        
        for item in tqdm(images_list, desc=f"{split}/{json_file}"):
            if item.get('is_box_issue') == 'Yes': continue
            
            save_dir = os.path.join(OUTPUT_BASE, split, item['final_emotion'])
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, item['filename'])
            
            # 이미 저장된 파일이면 빠르게 스킵 (재실행 시 유용)
            if os.path.exists(save_path): 
                success_count += 1
                continue
                
            # 1. 원본 이미지 로드
            img_path = os.path.join(img_base, item['final_emotion'], item['filename'])
            image = cv2.imread(img_path)
            if image is None: 
                error_count += 1
                continue
                
            # 2. NPZ 파일 찾기 (neutral일 경우 blank.npz도 확인)
            orig_emo = item['original_emotion']
            npz_paths_to_try = [
                os.path.join(npz_base, f"{split}_{orig_emo}.npz"),
                os.path.join(npz_base, f"{split}_blank.npz") if orig_emo == 'neutral' else ""
            ]
            
            npz_full_path = None
            for p in npz_paths_to_try:
                if p and os.path.exists(p):
                    npz_full_path = p
                    break
                    
            if not npz_full_path:
                error_count += 1
                continue
                
            # 3. NPZ 로드 (RAM 초과 방지를 위한 Lazy Load)
            if current_npz_path != npz_full_path:
                if npz_data is not None:
                    npz_data.close() # 이전 파일 메모리 해제
                current_npz_path = npz_full_path
                npz_data = np.load(npz_full_path)
                
            # 4. 마스크 Key 동적 탐색 (모든 가능한 이름의 경우의 수 확인)
            mask = None
            fname = item['filename']
            possible_keys = [fname, fname + ".npy", fname.replace('.jpg', '')]
            
            for k in possible_keys:
                if k in npz_data:
                    mask = npz_data[k]
                    break
                    
            if mask is None:
                error_count += 1
                continue
                
            # 5. 크롭 수행 및 배경 제거
            mask_3d = np.expand_dims(mask, axis=-1)
            face_only_img = (image * mask_3d).astype(np.uint8)
            
            box = item['bbox']
            x1, y1, x2, y2 = box['minX'], box['minY'], box['maxX'], box['maxY']
            h, w = image.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            if x2 <= x1 or y2 <= y1:
                error_count += 1
                continue
                
            face_cropped = face_only_img[y1:y2, x1:x2]
            face_resized = cv2.resize(face_cropped, (224, 224))
            
            if cv2.imwrite(save_path, face_resized):
                success_count += 1
            else:
                error_count += 1

if npz_data is not None:
    npz_data.close()

print(f"\n✅ 완료! 성공적으로 저장된 이미지: {success_count}장 / 실패 및 스킵: {error_count}장")'''

✂️ 데이터 영구 크롭 재작업 시작 (정밀 추적 모드)...


val/val_neutral.json:  50%|█████     | 63/126 [00:12<00:10,  6.25it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
val/val_anger.json: 100%|██████████| 184/184 [00:31<00:00,  5.89it/s]


✅ 완료! 성공적으로 저장된 이미지: 7045장 / 실패 및 스킵: 0장
